# Convert Hourly Temperature Data to Daily Temperature Data

This notebook converts hourly temperature data files (with lat/long coordinates) to daily temperature files.

**Process:**
1. Find all hourly temperature files with lat/long coordinates
2. Read each hourly file and extract date from MESS_DATUM
3. Filter out invalid data: -999 values and QN_9 < 3
4. Aggregate hourly temperatures to daily (mean, min, max)
5. Create new daily temperature files in the same folder structure
6. Save files to `/mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv/`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import traceback


In [ ]:
# Configuration
INPUT_BASE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv")
OUTPUT_BASE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv")

print(f"Input directory: {INPUT_BASE_DIR}")
print(f"Input directory exists: {INPUT_BASE_DIR.exists()}")
print(f"Output directory: {OUTPUT_BASE_DIR}")
print(f"Output directory exists: {OUTPUT_BASE_DIR.exists()}")

# Create output directory if it doesn't exist
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Note: Output files will be saved maintaining the same folder structure as input
# Input: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv/{station_dir}/{file}.csv
# Output: /mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv/{station_dir}/{file}.csv


Input directory: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv
Input directory exists: True
Output directory: /mnt/d/heatpump_data/climate_data/dwd_historical_daily_temperature_extracted_csv
Output directory exists: False


In [ ]:
def convert_hourly_to_daily(hourly_df):
    """
    Convert hourly temperature data to daily aggregated data.
    
    Args:
        hourly_df: DataFrame with hourly temperature data containing MESS_DATUM, TT_TU columns
        
    Returns:
        DataFrame with daily aggregated temperature data
    """
    if len(hourly_df) == 0:
        return pd.DataFrame()
    
    # Create a copy to avoid modifying original
    df = hourly_df.copy()
    
    # Extract date from MESS_DATUM (first 8 characters: YYYYMMDD)
    df['date'] = pd.to_datetime(
        df['MESS_DATUM'].astype(str).str[:8], 
        format='%Y%m%d',
        errors='coerce'
    )
    
    # Remove rows with invalid dates
    df = df[df['date'].notna()].copy()
    
    if len(df) == 0:
        return pd.DataFrame()
    
    # Get station metadata columns (should be constant per file)
    station_id = df['STATIONS_ID'].iloc[0] if 'STATIONS_ID' in df.columns else None
    latitude = df['latitude'].iloc[0] if 'latitude' in df.columns else None
    longitude = df['longitude'].iloc[0] if 'longitude' in df.columns else None
    
    # Convert TT_TU to numeric once upfront
    if 'TT_TU' in df.columns:
        df['TT_TU'] = pd.to_numeric(df['TT_TU'], errors='coerce')
        # Convert -999 (DWD missing value indicator) to NaN
        df['TT_TU'] = df['TT_TU'].replace(-999, np.nan)
    
    # Convert QN_9 to numeric once upfront
    if 'QN_9' in df.columns:
        df['QN_9'] = pd.to_numeric(df['QN_9'], errors='coerce')
    
    # Filter out invalid data:
    # 1. Remove rows where TT_TU is NaN (includes -999 after replacement)
    # 2. Remove rows where QN_9 < 3 (low quality data)
    valid_mask = df['TT_TU'].notna()
    
    if 'QN_9' in df.columns:
        # Keep rows where QN_9 is NaN (if quality flag doesn't exist) OR QN_9 >= 3
        qn_mask = df['QN_9'].isna() | (df['QN_9'] >= 3)
        valid_mask = valid_mask & qn_mask
    
    df_filtered = df[valid_mask].copy()
    
    if len(df_filtered) == 0:
        return pd.DataFrame()
    
    # Build aggregation dictionary
    agg_dict = {}
    
    if 'TT_TU' in df_filtered.columns:
        agg_dict['TT_TU'] = ['mean', 'min', 'max', 'count']
    
    if 'QN_9' in df_filtered.columns:
        # Use mode if available, otherwise mean
        def qn_agg(x):
            mode_vals = x.mode()
            if len(mode_vals) > 0:
                return mode_vals.iloc[0]
            else:
                return x.mean()
        agg_dict['QN_9'] = qn_agg
    
    # Perform aggregation using vectorized operations
    if agg_dict:
        daily_df = df_filtered.groupby('date').agg(agg_dict)
        
        # Flatten column names
        if 'TT_TU' in agg_dict and 'QN_9' in agg_dict:
            daily_df.columns = ['TT_TU_mean', 'TT_TU_min', 'TT_TU_max', 'TT_TU_count', 'QN_9']
        elif 'TT_TU' in agg_dict:
            daily_df.columns = ['TT_TU_mean', 'TT_TU_min', 'TT_TU_max', 'TT_TU_count']
        else:
            daily_df.columns = ['QN_9']
        
        # Reset index to get date as column
        daily_df = daily_df.reset_index()
    else:
        # If no aggregations, just group by date
        daily_df = df_filtered.groupby('date').size().reset_index(name='count')
    
    # Format date as YYYYMMDD
    daily_df['MESS_DATUM'] = daily_df['date'].dt.strftime('%Y%m%d')
    
    # Add station metadata
    daily_df['STATIONS_ID'] = station_id
    daily_df['latitude'] = latitude
    daily_df['longitude'] = longitude
    
    # Reorder columns
    cols = ['STATIONS_ID', 'MESS_DATUM', 'latitude', 'longitude']
    if 'TT_TU_mean' in daily_df.columns:
        cols.extend(['TT_TU_mean', 'TT_TU_min', 'TT_TU_max', 'TT_TU_count'])
    if 'QN_9' in daily_df.columns:
        cols.append('QN_9')
    
    daily_df = daily_df[cols]
    
    # Sort by date
    daily_df = daily_df.sort_values('MESS_DATUM').reset_index(drop=True)
    
    return daily_df


In [ ]:
def process_hourly_file(hourly_file_path, output_base_dir):
    """
    Process a single hourly temperature file and create daily temperature file.
    
    Args:
        hourly_file_path: Path to hourly temperature CSV file
        output_base_dir: Base directory for output files
        
    Returns:
        Dictionary with processing results
    """
    try:
        # Read hourly file
        hourly_df = pd.read_csv(hourly_file_path)
        
        if len(hourly_df) == 0:
            return {
                'status': 'skipped',
                'reason': 'Empty file',
                'file': hourly_file_path.name
            }
        
        # Convert to daily
        daily_df = convert_hourly_to_daily(hourly_df)
        
        if len(daily_df) == 0:
            return {
                'status': 'skipped',
                'reason': 'No valid dates',
                'file': hourly_file_path.name
            }
        
        # Get relative path from input base directory
        relative_path = hourly_file_path.relative_to(INPUT_BASE_DIR)
        
        # Create output path maintaining same folder structure
        output_file_path = output_base_dir / relative_path
        
        # Create parent directories if they don't exist
        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save daily data
        daily_df.to_csv(output_file_path, index=False)
        
        return {
            'status': 'success',
            'file': hourly_file_path.name,
            'output_file': output_file_path.name,
            'hourly_records': len(hourly_df),
            'daily_records': len(daily_df),
            'station_id': daily_df['STATIONS_ID'].iloc[0] if len(daily_df) > 0 else None
        }
        
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e),
            'file': hourly_file_path.name,
            'traceback': traceback.format_exc()
        }


## Test: Process a Single File

Test the conversion process on a single file before processing all files.


In [ ]:
# Test: Process a single file for assessment
print("=" * 80)
print("TEST: Processing a single file")
print("=" * 80)

# Find one hourly file with lat/long coordinates
test_files = list(INPUT_BASE_DIR.rglob("*_lat_*_lon_*.csv"))
if len(test_files) == 0:
    print("No files found with lat/long coordinates!")
else:
    # Select the first file with data (or just the first file)
    test_file = test_files[0]
    print(f"\nTest file: {test_file.name}")
    print(f"Full path: {test_file}")
    
    # Read and display sample of hourly data
    print("\n" + "-" * 80)
    print("HOURLY DATA (Input)")
    print("-" * 80)
    hourly_df = pd.read_csv(test_file)
    print(f"Total hourly records: {len(hourly_df):,}")
    print(f"Columns: {', '.join(hourly_df.columns)}")
    print(f"\nFirst 5 rows:")
    print(hourly_df.head())
    print(f"\nLast 5 rows:")
    print(hourly_df.tail())
    
    # Convert to daily
    print("\n" + "-" * 80)
    print("CONVERTING TO DAILY...")
    print("-" * 80)
    daily_df = convert_hourly_to_daily(hourly_df)
    
    # Display daily data
    print("\n" + "-" * 80)
    print("DAILY DATA (Output)")
    print("-" * 80)
    print(f"Total daily records: {len(daily_df):,}")
    print(f"Columns: {', '.join(daily_df.columns)}")
    print(f"\nFirst 10 rows:")
    print(daily_df.head(10))
    print(f"\nLast 10 rows:")
    print(daily_df.tail(10))
    
    # Statistics
    print("\n" + "-" * 80)
    print("STATISTICS")
    print("-" * 80)
    if len(daily_df) > 0:
        print(f"Date range: {daily_df['MESS_DATUM'].min()} to {daily_df['MESS_DATUM'].max()}")
        print(f"Station ID: {daily_df['STATIONS_ID'].iloc[0]}")
        print(f"Latitude: {daily_df['latitude'].iloc[0]}")
        print(f"Longitude: {daily_df['longitude'].iloc[0]}")
        if 'TT_TU_mean' in daily_df.columns:
            print(f"\nTemperature statistics:")
            print(f"  Mean daily temp: {daily_df['TT_TU_mean'].mean():.2f}°C")
            print(f"  Min daily temp: {daily_df['TT_TU_min'].min():.2f}°C")
            print(f"  Max daily temp: {daily_df['TT_TU_max'].max():.2f}°C")
            print(f"  Avg hours per day: {daily_df['TT_TU_count'].mean():.1f}")
    
    # Test saving the file
    print("\n" + "-" * 80)
    print("TESTING FILE SAVE...")
    print("-" * 80)
    result = process_hourly_file(test_file, OUTPUT_BASE_DIR)
    print(f"Status: {result['status']}")
    if result['status'] == 'success':
        relative_path = test_file.relative_to(INPUT_BASE_DIR)
        output_file_path = OUTPUT_BASE_DIR / relative_path
        print(f"Output file saved to: {output_file_path}")
        print(f"Output file exists: {output_file_path.exists()}")
        if output_file_path.exists():
            file_size_kb = output_file_path.stat().st_size / 1024
            print(f"Output file size: {file_size_kb:.2f} KB")
            print(f"Hourly records: {result.get('hourly_records', 0):,}")
            print(f"Daily records: {result.get('daily_records', 0):,}")
            print(f"Compression ratio: {result.get('hourly_records', 1) / max(result.get('daily_records', 1), 1):.1f}:1")
    else:
        print(f"Error/Skip reason: {result.get('reason', result.get('error', 'Unknown'))}")
    
    print("\n" + "=" * 80)
    print("TEST COMPLETE")
    print("=" * 80)


TEST: Processing a single file

Test file: produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
Full path: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv/stundenwerte_TU_00003_19500401_20110331_hist/produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv

--------------------------------------------------------------------------------
HOURLY DATA (Input)
--------------------------------------------------------------------------------
Total hourly records: 534,719
Columns: STATIONS_ID, MESS_DATUM, QN_9, TT_TU, RF_TU, eor, latitude, longitude

First 5 rows:
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor  latitude  longitude
0            3  1950040101     5    5.7   83.0  eor   50.7827     6.0941
1            3  1950040102     5    5.6   83.0  eor   50.7827     6.0941
2            3  1950040103     5    5.5   83.0  eor   50.7827     6.0941
3            3  1950040104     5    5.5   83.0  eor   50.7827     6.0941
4        

In [ ]:
# Find all hourly temperature files with lat/long coordinates
print("=" * 80)
print("Finding all hourly temperature files with lat/long coordinates...")
print("=" * 80)

hourly_files = list(INPUT_BASE_DIR.rglob("*_lat_*_lon_*.csv"))
hourly_files.sort()

print(f"Found {len(hourly_files)} hourly temperature files with lat/long coordinates")


Finding all hourly temperature files with lat/long coordinates...
Found 2796 hourly temperature files with lat/long coordinates


In [ ]:
# Process all hourly files and convert to daily
print("=" * 80)
print("Processing hourly files and converting to daily temperatures...")
print("=" * 80)

results = []
success_count = 0
error_count = 0
skipped_count = 0

for hourly_file in tqdm(hourly_files, desc="Processing files", unit="file"):
    result = process_hourly_file(hourly_file, OUTPUT_BASE_DIR)
    result['input_file_path'] = str(hourly_file)
    results.append(result)
    
    if result['status'] == 'success':
        success_count += 1
    elif result['status'] == 'error':
        error_count += 1
    elif result['status'] == 'skipped':
        skipped_count += 1

print("\n" + "=" * 80)
print("Processing complete!")
print("=" * 80)


Processing hourly files and converting to daily temperatures...


Processing files:   0%|          | 0/2796 [00:00<?, ?file/s]

Processing files: 100%|██████████| 2796/2796 [13:45<00:00,  3.39file/s]


Processing complete!


In [25]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(f"\nTotal files processed: {len(results)}")
print(f"  Successfully processed: {success_count}")
print(f"  Errors: {error_count}")
print(f"  Skipped: {skipped_count}")

# Aggregate statistics from successful processing
successful_results = [r for r in results if r['status'] == 'success']

if successful_results:
    total_hourly_records = sum(r.get('hourly_records', 0) for r in successful_results)
    total_daily_records = sum(r.get('daily_records', 0) for r in successful_results)
    unique_stations = len(set(r.get('station_id') for r in successful_results if r.get('station_id') is not None))
    
    print(f"\nData conversion:")
    print(f"  Total hourly records processed: {total_hourly_records:,}")
    print(f"  Total daily records created: {total_daily_records:,}")
    print(f"  Unique stations: {unique_stations}")
    print(f"  Average records per file: {total_daily_records / len(successful_results):.1f}")

# Show files with errors
error_results = [r for r in results if r['status'] == 'error']
if error_results:
    print(f"\n{'=' * 80}")
    print(f"FILES WITH ERRORS ({len(error_results)}):")
    print("=" * 80)
    for r in error_results[:10]:  # Show first 10 errors
        print(f"\nFile: {r['file']}")
        print(f"Error: {r['error']}")
    if len(error_results) > 10:
        print(f"\n... and {len(error_results) - 10} more errors")



SUMMARY STATISTICS

Total files processed: 2796
  Successfully processed: 1005
  Errors: 0
  Skipped: 1791

Data conversion:
  Total hourly records processed: 139,884,180
  Total daily records created: 5,852,162
  Unique stations: 1005
  Average records per file: 5823.0


Checking for -999 values in hourly temperature data

Station directory: stundenwerte_TU_00078_20041101_20241231_hist
Full path: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv/stundenwerte_TU_00078_20041101_20241231_hist

Found 4 files in this station directory

Checking file: produkt_tu_stunde_20041101_20241231_78_1_lat_52_4671_lon_7_938.csv
  File is empty

Checking file: produkt_tu_stunde_20041101_20241231_78_2_lat_52_4787_lon_7_938.csv
  File is empty

Checking file: produkt_tu_stunde_20041101_20241231_78_3_lat_52_4853_lon_7_9125.csv
  File is empty

Checking file: produkt_tu_stunde_20041101_20241231_78_4_lat_52_5026_lon_7_9468.csv
  Total records: 176,781
  Records with TT_TU = -999: 23 (0.01%)
  Records with TT_TU = NaN: 0

  Sample rows with -999:
  First 5 occurrences:
       STATIONS_ID  MESS_DATUM  TT_TU  latitude  longitude
154256        78_4  2022060711 -999.0   52.5026     7.9468
154257        78_4  2022060712 -999.0   52.5026     7.9468
1

/tmp/ipykernel_1030/348588729.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  minus_999_rows['date'] = pd.to_datetime(


Checking Metadaten_Fehlwerte files for missing value documentation

Station directory: stundenwerte_TU_00078_20041101_20241231_hist
Full path: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv/stundenwerte_TU_00078_20041101_20241231_hist

Found 3 Metadaten_Fehlwerte file(s):
  - Metadaten_Fehlwerte_00078_20041101_20241231.csv
  - Metadaten_Fehldaten_00078_20041101_20241231.csv
  - Metadaten_Fehlwerte_00078_20041101_20241231.csv

Reading file: Metadaten_Fehlwerte_00078_20041101_20241231.csv

File shape: (9, 9)
Columns: Stations_ID, Stations_Name, Parameter, Von_Datum, Bis_Datum, Anzahl_Fehlwerte, Beschreibung, eor,                                                                                                               

First few rows:
                                         Stations_ID Stations_Name Parameter  \
0                                                 78     Alfhausen     TT_TU   
1                                                 78     A

In [26]:
# Sample verification - check a few converted files
print("\n" + "=" * 80)
print("SAMPLE VERIFICATION")
print("=" * 80)

sample_results = [r for r in successful_results[:3]]

for result in sample_results:
    # Get output file path
    relative_path = Path(result['input_file_path']).relative_to(INPUT_BASE_DIR)
    output_file_path = OUTPUT_BASE_DIR / relative_path
    
    print(f"\nInput file: {result['file']}")
    print(f"  Hourly records: {result.get('hourly_records', 0):,}")
    print(f"  Daily records: {result.get('daily_records', 0):,}")
    print(f"  Station ID: {result.get('station_id', 'N/A')}")
    
    if output_file_path.exists():
        df_check = pd.read_csv(output_file_path)
        print(f"  Output file: {output_file_path.name}")
        print(f"  Output file size: {output_file_path.stat().st_size / 1024:.1f} KB")
        print(f"  Columns: {', '.join(df_check.columns)}")
        if len(df_check) > 0:
            print(f"  Date range: {df_check['MESS_DATUM'].min()} to {df_check['MESS_DATUM'].max()}")
            print(f"  Sample row:")
            print(f"    {df_check.iloc[0].to_dict()}")



SAMPLE VERIFICATION

Input file: produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
  Hourly records: 534,719
  Daily records: 22,280
  Station ID: 3
  Output file: produkt_tu_stunde_19500401_20110331_00003_lat_50_7827_lon_6_0941.csv
  Output file size: 1233.3 KB
  Columns: STATIONS_ID, MESS_DATUM, latitude, longitude, TT_TU_mean, TT_TU_min, TT_TU_max, TT_TU_count, QN_9
  Date range: 19500401 to 20110331
  Sample row:
    {'STATIONS_ID': 3.0, 'MESS_DATUM': 19500401.0, 'latitude': 50.7827, 'longitude': 6.0941, 'TT_TU_mean': 8.034782608695652, 'TT_TU_min': 5.5, 'TT_TU_max': 10.1, 'TT_TU_count': 23.0, 'QN_9': 5.0}

Input file: produkt_tu_stunde_19500401_20110331_3_2_lat_50_7827_lon_6_0941.csv
  Hourly records: 534,719
  Daily records: 22,280
  Station ID: 3_2
  Output file: produkt_tu_stunde_19500401_20110331_3_2_lat_50_7827_lon_6_0941.csv
  Output file size: 1276.8 KB
  Columns: STATIONS_ID, MESS_DATUM, latitude, longitude, TT_TU_mean, TT_TU_min, TT_TU_max, TT_TU_count